A. Membangun Struktur Direktori HDFS

In [5]:
# Membuat direktori raw dan processed di HDFS
!hdfs dfs -mkdir -p /user/fathan/ecommerce/raw
!hdfs dfs -mkdir -p /user/fathan/ecommerce/processed

# Verifikasi struktur direktori
!hdfs dfs -ls -R /user/fathan/ecommerce

drwxr-xr-x   - pathan supergroup          0 2026-09-17 09:10 /user/fathan/ecommerce/processed
drwxr-xr-x   - pathan supergroup          0 2026-09-17 09:10 /user/fathan/ecommerce/raw


B. Mengunggah Data Mentah ke HDFS

In [6]:
# Upload ketiga berkas CSV ke folder raw di HDFS
!hdfs dfs -put transaksi_magelang.csv /user/fathan/ecommerce/raw/
!hdfs dfs -put transaksi_yogyakarta.csv /user/fathan/ecommerce/raw/
!hdfs dfs -put transaksi_semarang.csv /user/fathan/ecommerce/raw/

# Verifikasi berkas yang sudah terunggah
!hdfs dfs -du -h /user/fathan/ecommerce/raw

12.0 K  12.0 K  /user/fathan/ecommerce/raw/transaksi_magelang.csv
11.9 K  11.9 K  /user/fathan/ecommerce/raw/transaksi_semarang.csv
12.4 K  12.4 K  /user/fathan/ecommerce/raw/transaksi_yogyakarta.csv


C. Membaca Kembali dan Menggabungkan Data dari HDFS 

In [7]:
import subprocess
import pandas as pd
import io

files = ["transaksi_magelang.csv", "transaksi_yogyakarta.csv", "transaksi_semarang.csv"]
dfs = []

# Membaca data langsung dari HDFS menggunakan pipeline cat
for file_name in files:
    hdfs_path = f"/user/fathan/ecommerce/raw/{file_name}"
    cmd = f"hdfs dfs -cat {hdfs_path}"
    output = subprocess.check_output(cmd, shell=True)
    df_temp = pd.read_csv(io.BytesIO(output))
    dfs.append(df_temp)

# Gabungkan ketiga DataFrame
df_gabungan = pd.concat(dfs, ignore_index=True)

# Tampilkan ringkasan data
print("Jumlah transaksi per kota:")
print(df_gabungan['kota'].value_counts())
df_gabungan.head()

Jumlah transaksi per kota:
kota
Magelang      200
Yogyakarta    200
Semarang      200
Name: count, dtype: int64


,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang


D. Mengolah dan Download Hasil ke Direktori processed

In [11]:
# 1. Hitung total pendapatan per transaksi
df_gabungan['total_pendapatan'] = df_gabungan['unit_terjual'] * df_gabungan['harga_satuan']

# 2. Buat ringkasan total pendapatan per kota dan per kategori
ringkasan = df_gabungan.groupby(['kota', 'kategori'])['total_pendapatan'].sum().reset_index()

# 3. Simpan hasil olahan ke disk
df_gabungan.to_csv("data_gabungan_bersih.csv", index=False)
ringkasan.to_csv("ringkasan_kota_kategori.csv", index=False)

print("Pembersihan dan olah data berhasil, berkas lokal siap diunggah.")

Pembersihan dan olah data berhasil, berkas lokal siap diunggah.


Unggah Hasil Olahan ke HDFS

In [10]:
# Unggah berkas olahan ke folder processed di HDFS
!hdfs dfs -put data_gabungan_bersih.csv /user/fathan/ecommerce/processed/
!hdfs dfs -put ringkasan_kota_kategori.csv /user/fathan/ecommerce/processed/

# Verifikasi isi folder processed
!hdfs dfs -ls -h /user/fathan/ecommerce/processed

Found 2 items
-rw-r--r--   1 pathan supergroup     40.3 K 2026-09-17 09:21 /user/fathan/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 pathan supergroup        530 2026-09-17 09:21 /user/fathan/ecommerce/processed/ringkasan_kota_kategori.csv
